# OfficeQA Retrieval Evaluation

Runs semantic top-k retrieval for every question in `data/officeqa/officeqa_pro.csv` using the same `SemTopKOperator` harness pattern as `run_one_retrieval.py`, then reports aggregate precision, recall, and F1.

In [ ]:
from __future__ import annotations

import csv
import glob
import os
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().parents[1]

sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

from carnot.data.dataset import Dataset
from carnot.operators.sem_topk import SemTopKOperator

if load_dotenv is not None:
    load_dotenv(REPO_ROOT / ".env")

QUERIES_CSV = REPO_ROOT / "data" / "officeqa" / "officeqa_pro.csv"
DOCS_DIR = REPO_ROOT / "data" / "officeqa" / "treasury_bulletins_parsed" / "transformed"

K = 5
INDEX = "faiss"
EMBEDDING_MODEL = "openai/text-embedding-3-small"
MAX_QUERIES = None
MAX_DOCS = None

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.max_columns", 40)

In [ ]:
with open(QUERIES_CSV, newline="") as queries_file:
    queries = list(csv.DictReader(queries_file))

if MAX_QUERIES is not None:
    queries = queries[:MAX_QUERIES]

document_paths = sorted(glob.glob(str(DOCS_DIR / "*.txt")))
if MAX_DOCS is not None:
    gold_files = []
    for query in queries:
        gold_files.extend(
            source_file.strip()
            for source_file in query["source_files"].splitlines()
            if source_file.strip()
        )
    gold_paths = [str(DOCS_DIR / source_file) for source_file in gold_files]
    document_paths = list(dict.fromkeys(gold_paths + document_paths))[:MAX_DOCS]
    document_paths = [path for path in document_paths if os.path.exists(path)]

if not document_paths:
    raise ValueError(f"No .txt documents found in {DOCS_DIR}")

items = []
for document_path in document_paths:
    with open(document_path, errors="replace") as document:
        items.append(
            {
                "uri": document_path,
                "source_file": os.path.basename(document_path),
                "contents": document.read(),
            }
        )

dataset = Dataset(
    name="OfficeQA Documents",
    annotation="Parsed Treasury Bulletin documents for OfficeQA retrieval.",
    items=items,
    dataset_id="officeqa_documents",
)

llm_config = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "GOOGLE_API_KEY": os.getenv("GOOGLE_API_KEY"),
    "GEMINI_API_KEY": os.getenv("GEMINI_API_KEY"),
}

print(f"Loaded {len(queries)} queries")
print(f"Loaded {len(items)} documents from {DOCS_DIR}")
print(f"Retrieval config: index={INDEX}, k={K}, embedding_model={EMBEDDING_MODEL}")

In [ ]:
def split_source_files(value: str) -> list[str]:
    return [source_file.strip() for source_file in value.splitlines() if source_file.strip()]


def precision_recall_f1(gold_files: list[str], predicted_files: list[str]) -> dict[str, float | int | bool]:
    gold = set(gold_files)
    predicted = {source_file for source_file in predicted_files if source_file}
    true_positives = len(gold & predicted)
    precision = true_positives / len(predicted) if predicted else 0.0
    recall = true_positives / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "true_positives": true_positives,
        "gold_count": len(gold),
        "predicted_count": len(predicted),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "hit_any": true_positives > 0,
    }


def retrieve_for_query(query: dict[str, str]) -> dict[str, object]:
    gold_files = split_source_files(query["source_files"])
    operator = SemTopKOperator(
        task=query["question"],
        k=K,
        dataset_id="RetrievedOfficeQADocuments",
        max_workers=1,
        index_name=INDEX,
        model_id=EMBEDDING_MODEL,
        llm_config=llm_config,
    )
    output_datasets, stats = operator(dataset.name, {dataset.name: dataset})
    retrieved = output_datasets["RetrievedOfficeQADocuments"].items
    predicted_files = [item.get("source_file") for item in retrieved]
    metrics = precision_recall_f1(gold_files, predicted_files)
    return {
        "uid": query["uid"],
        "difficulty": query.get("difficulty"),
        "question": query["question"],
        "answer": query.get("answer"),
        "gold_source_files": gold_files,
        "predicted_source_files": predicted_files,
        "items_in": stats.items_in,
        "items_out": stats.items_out,
        "error": None,
        **metrics,
    }

In [ ]:
rows = []
for idx, query in enumerate(queries, start=1):
    try:
        rows.append(retrieve_for_query(query))
    except Exception as exc:
        gold_files = split_source_files(query["source_files"])
        rows.append(
            {
                "uid": query["uid"],
                "difficulty": query.get("difficulty"),
                "question": query["question"],
                "answer": query.get("answer"),
                "gold_source_files": gold_files,
                "predicted_source_files": [],
                "items_in": len(items),
                "items_out": 0,
                "error": repr(exc),
                **precision_recall_f1(gold_files, []),
            }
        )
    if idx == 1 or idx % 10 == 0 or idx == len(queries):
        print(f"Completed {idx}/{len(queries)} queries")

results_df = pd.DataFrame(rows)
results_df.head()

In [ ]:
total_tp = int(results_df["true_positives"].sum())
total_predicted = int(results_df["predicted_count"].sum())
total_gold = int(results_df["gold_count"].sum())
micro_precision = total_tp / total_predicted if total_predicted else 0.0
micro_recall = total_tp / total_gold if total_gold else 0.0
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if micro_precision + micro_recall else 0.0

aggregate_metrics = pd.DataFrame(
    [
        {
            "aggregation": "macro_avg_across_queries",
            "queries": len(results_df),
            "precision": results_df["precision"].mean(),
            "recall": results_df["recall"].mean(),
            "f1": results_df["f1"].mean(),
            "hit_rate": results_df["hit_any"].mean(),
            "true_positives": total_tp,
            "gold_files": total_gold,
            "predicted_files": total_predicted,
            "failed_queries": int(results_df["error"].notna().sum()),
            "k": K,
            "index": INDEX,
        },
        {
            "aggregation": "micro_total_across_files",
            "queries": len(results_df),
            "precision": micro_precision,
            "recall": micro_recall,
            "f1": micro_f1,
            "hit_rate": results_df["hit_any"].mean(),
            "true_positives": total_tp,
            "gold_files": total_gold,
            "predicted_files": total_predicted,
            "failed_queries": int(results_df["error"].notna().sum()),
            "k": K,
            "index": INDEX,
        },
    ]
)

aggregate_metrics.style.format(
    {
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "hit_rate": "{:.4f}",
    }
)

In [ ]:
best_result = results_df.sort_values(
    ["f1", "recall", "precision", "true_positives"],
    ascending=[False, False, False, False],
).head(1)

best_result[
    [
        "uid",
        "difficulty",
        "precision",
        "recall",
        "f1",
        "hit_any",
        "question",
        "gold_source_files",
        "predicted_source_files",
        "answer",
        "error",
    ]
]

In [ ]:
worst_result = results_df.sort_values(
    ["f1", "recall", "precision", "true_positives"],
    ascending=[True, True, True, True],
).head(1)

worst_result[
    [
        "uid",
        "difficulty",
        "precision",
        "recall",
        "f1",
        "hit_any",
        "question",
        "gold_source_files",
        "predicted_source_files",
        "answer",
        "error",
    ]
]